<a href="https://colab.research.google.com/github/julianema16/Grafico-3d-Flecha/blob/main/Grafico_3D_en_el_espacio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# 1. Definición de Arrays
posiciones = [f"F{i}" for i in range(14)]
xc = np.array([0.716, -0.08, -0.412, -0.851, -0.915, -1.022, -0.664, -0.424, -0.302, 0.226, 0.45, 0.906, 1.328, 1.935])
yc = np.array([1.574, 2.097, 2.502, 2.547, 3.004, 3.25, 3.831, 4.091, 4.296, 4.662, 4.84, 4.907, 4.965, 4.957])
z_abs = np.array([5794, 5290, 4835, 4380, 3925, 3470, 3015, 2560, 2105, 1650, 1195, 795, 456, 295])
puntos = np.column_stack((xc, yc, z_abs))

# 2. Geometría: Recta Teórica y Distancias Ortogonales
p0, p13 = puntos[0], puntos[-1]
vector_director = p13 - p0
vector_norm_sq = np.dot(vector_director, vector_director)

puntos_proyectados = []
distancias = []

for p in puntos:
    w = p - p0
    t = np.dot(w, vector_director) / vector_norm_sq
    proyeccion = p0 + t * vector_director
    puntos_proyectados.append(proyeccion)
    distancias.append(np.linalg.norm(p - proyeccion))

puntos_proyectados = np.array(puntos_proyectados)

# 3. Creación del Layout Dividido (1 Fila, 2 Columnas)
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "table"}, {"type": "scene"}]],
    column_widths=[0.45, 0.55],
    horizontal_spacing=0.02
)

# 4. Traza de Tabla de Datos (Columna 1)
fig.add_trace(go.Table(
    header=dict(
        values=["<b>Pos</b>", "<b>Xc Real</b>", "<b>Yc Real</b>", "<b>Z Real</b>",
                "<b>X Recta</b>", "<b>Y Recta</b>", "<b>Z Recta</b>", "<b>Vector Desvío</b>"],
        fill_color='midnightblue',
        font=dict(color='white', size=11),
        align="center"
    ),
    cells=dict(
        values=[
            posiciones,
            np.round(xc, 3), np.round(yc, 3), z_abs,
            np.round(puntos_proyectados[:, 0], 3),
            np.round(puntos_proyectados[:, 1], 3),
            np.round(puntos_proyectados[:, 2], 1),
            np.round(distancias, 3)
        ],
        fill_color='aliceblue',
        align="center",
        font=dict(size=10)
    )
), row=1, col=1)

# 5. Trazas 3D (Columna 2)
# Eje Real
fig.add_trace(go.Scatter3d(
    x=xc, y=yc, z=z_abs, mode='lines+markers+text', text=posiciones,
    marker=dict(size=4, color=distancias, colorscale='Reds'),
    line=dict(color='darkblue', width=3), name='Eje Físico'
), row=1, col=2)

# Eje Teórico
fig.add_trace(go.Scatter3d(
    x=[p0[0], p13[0]], y=[p0[1], p13[1]], z=[p0[2], p13[2]], mode='lines',
    line=dict(color='red', width=2, dash='dash'), name='Eje Ideal'
), row=1, col=2)

# Vectores de Error
for i in range(1, 13):
    fig.add_trace(go.Scatter3d(
        x=[puntos[i][0], puntos_proyectados[i][0]],
        y=[puntos[i][1], puntos_proyectados[i][1]],
        z=[puntos[i][2], puntos_proyectados[i][2]],
        mode='lines', line=dict(color='orange', width=2),
        showlegend=False
    ), row=1, col=2)

# 6. Configuración Final
fig.update_layout(
    title="Dashboard de Torsión: Análisis Tabular y Espacial",
    scene=dict(
        xaxis_title='Xc (mm)', yaxis_title='Yc (mm)', zaxis_title='Z (mm)',
        aspectmode='manual', aspectratio=dict(x=1.5, y=1.5, z=4)
    ),
    margin=dict(l=10, r=10, b=10, t=50)
)

fig.write_html("Dashboard_Control_Flecha.html")
fig.show()